# M2 · E4 SAHI dengan bobot E2 historis di Kaggle

Tiga alur memakai satu checkpoint E2 native 1280 yang berasal dari pelatihan historis lima epoch. Semua alur mengevaluasi 356 gambar validasi dan 2.205 kotak acuan yang sama. Kontrol menginferensi satu gambar utuh pada 1280. Dua varian SAHI memakai tile 640 dan overlap nominal 20%, dengan opsi prediksi standar pada gambar utuh dinonaktifkan atau diaktifkan. Saat diaktifkan, SAHI memproses gambar utuh pada input 640 yang sama dengan tile, lalu menggabungkan prediksi.

Skor dihitung dari prediksi CPU pada koordinat acuan 640. Latensi dihitung ulang pada satu CPU dengan empat thread, 10 gambar pemanasan, dan 100 gambar validasi yang sama untuk seluruh alur. Waktu baca JPEG berada di luar pengukuran.

In [ ]:
import os
import subprocess
import sys
os.environ["YOLO_CONFIG_DIR"] = "/tmp/ciringoni-yolo"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics==8.4.210", "pycocotools==2.0.11", "sahi==0.14.0"],
    check=True,
)

In [ ]:
import hashlib
import json
import tarfile
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve

SOURCE_COMMIT = "ee8c14bd132a07db3dae4bb15de0d8a22f517d60"
E2_SHA256 = "0bd467aef478ebd26555074030ff653c52cf05ad5380fcbe66057b3c995f264e"

def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

scratch = TemporaryDirectory(prefix="ciringoni-e4-")
project = Path(scratch.name) / "Data_Mining_Ciringoni_M1"
project.mkdir()
input_root = Path("/kaggle/input")
archives = sorted(input_root.rglob("validation_e2.tar.gz"))
if archives:
    assert len(archives) == 1
    with tarfile.open(archives[0]) as archive:
        archive.extractall(project, filter="data")
    data = project
else:
    manifests = sorted(input_root.rglob("geometry.json"))
    assert len(manifests) == 1, "Lampirkan dataset privat validasi E2."
    data = manifests[0].parent

gt = data / "hasil_m1/coco_val_gt.json"
ground_truth = json.loads(gt.read_text())
assert (len(ground_truth["images"]), len(ground_truth["annotations"])) == (356, 2205)
checkpoint = data / "1280/best.pt"
assert sha256(checkpoint) == E2_SHA256
manifest = data / "geometry.json"
assert json.loads(manifest.read_text())["ready"]

gt_for_evaluator = project / "hasil_m1/coco_val_gt.json"
gt_for_evaluator.parent.mkdir(exist_ok=True)
if gt != gt_for_evaluator:
    gt_for_evaluator.write_bytes(gt.read_bytes())
sources = [
    "hasil_m2/evaluasi_e4_sahi.py", "hasil_m2/evaluasi_e2.py",
    "hasil_m2/evaluasi_mosaic.py", "hasil_m2/uji_tiling_640.py",
]
source_files = {}
for name in sources:
    destination = project / name
    destination.parent.mkdir(exist_ok=True)
    urlretrieve(
        f"https://raw.githubusercontent.com/Jordyee/ciringoni-data-mining/{SOURCE_COMMIT}/Data_Mining_Ciringoni_M1/{name}",
        destination,
    )
    source_files[name] = sha256(destination)
print("Validation: 356 images, 2205 boxes. E2 checkpoint SHA-256:", E2_SHA256)
print("Source commit:", SOURCE_COMMIT)
from importlib.metadata import version
from IPython.display import Image, display
import cv2
import numpy as np
import pandas as pd
import torch

native = data
audits = (manifest,)
gt_path = data / "hasil_m1/coco_val_gt.json"
code_hashes = source_files
sys.path.insert(0, str(project / "hasil_m2"))
affinity = sorted(os.sched_getaffinity(0))[:4]
assert len(affinity) == 4
os.sched_setaffinity(0, affinity)
cv2.setNumThreads(1)
torch.set_num_threads(4)


In [ ]:
from collections import defaultdict
from random import Random
from time import perf_counter
from pycocotools.coco import COCO
from sahi import AutoDetectionModel
from sahi.predict import get_prediction, get_sliced_prediction
from sahi.slicing import get_slice_bboxes

from evaluasi_e4_sahi import read_rgb, rows_for_coco, write_evidence
from evaluasi_mosaic import precision_recall_fixed
from uji_tiling_640 import score

TILE = 640
OVERLAP = 0.20
MERGE_IOU = 0.5
CONFIDENCE = 0.001
SLICE_BATCH = 4
ARM_NAMES = ("full_native_1280", "sahi_tiles", "sahi_tiles_and_standard")

def infer(model, rgb, arm):
    if arm == "full_native_1280":
        model.image_size = 1280
        return get_prediction(rgb, model)
    model.image_size = TILE
    return get_sliced_prediction(
        rgb, model, slice_height=TILE, slice_width=TILE,
        overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP,
        perform_standard_pred=arm == "sahi_tiles_and_standard",
        postprocess_type="NMS", postprocess_match_metric="IOU",
        postprocess_match_threshold=MERGE_IOU, postprocess_class_agnostic=False,
        verbose=0, batch_size=SLICE_BATCH,
    )

def load_model(device):
    return AutoDetectionModel.from_pretrained(
        model_type="ultralytics", model_path=str(checkpoint),
        confidence_threshold=CONFIDENCE, device=device,
    )

gt = COCO(str(gt_for_evaluator))
ids = sorted(gt.imgs)
assert ids == list(range(1, 357)) and len(gt.anns) == 2205
audit = json.loads(audits[0].read_text())
assert audit["ready"]
geometry = {Path(row["filename"]).stem: row for row in audit["files"] if row["split"] == "val" and row["status"] == "paired"}
image_paths = {i: native / "1280/images/val" / gt.imgs[i]["file_name"] for i in ids}
assert len(geometry) == 356 and {p.stem for p in image_paths.values()} == set(geometry)
assert set((native / "1280/images/val").glob("*.jpg")) == set(image_paths.values())
detection = load_model("cpu")
for i in ids[:2]:
    path = image_paths[i]
    rgb = read_rgb(path)
    for arm in ARM_NAMES:
        rows_for_coco(infer(detection, rgb, arm), i, geometry[path.stem])
print("Smoke: three arms, two images.")
predictions = defaultdict(list)
for index, i in enumerate(ids, 1):
    path = image_paths[i]
    rgb = read_rgb(path)
    for arm in ARM_NAMES:
        predictions[arm].extend(rows_for_coco(infer(detection, rgb, arm), i, geometry[path.stem]))
    if index % 50 == 0 or index == len(ids):
        print(f"{index}/{len(ids)} validation images", flush=True)
del detection


In [ ]:
available = sorted(os.sched_getaffinity(0))
assert len(available) >= 4, available
affinity = available[:4]
os.sched_setaffinity(0, affinity)
torch.set_num_threads(4)
cv2.setNumThreads(1)
cpu_model = load_model("cpu")
sample = Random(42).sample(ids, 110)
for i in sample[:10]:
    rgb = read_rgb(image_paths[i])
    for arm in ARM_NAMES:
        infer(cpu_model, rgb, arm)
torch.set_num_threads(4)
latency = {arm: [] for arm in ARM_NAMES}
for index, i in enumerate(sample[10:]):
    path = image_paths[i]
    rgb = read_rgb(path)
    offset = index % len(ARM_NAMES)
    for arm in ARM_NAMES[offset:] + ARM_NAMES[:offset]:
        start = perf_counter()
        result = infer(cpu_model, rgb, arm)
        rows_for_coco(result, i, geometry[path.stem])
        latency[arm].append({"image_id": i, "ms": (perf_counter() - start) * 1000})
    if (index + 1) % 25 == 0:
        print(f"{index + 1}/100 CPU benchmark images", flush=True)
assert torch.get_num_threads() == 4
del cpu_model

In [ ]:
output = Path("/kaggle/working/E4")
output.mkdir()
rows = []
for arm in ARM_NAMES:
    detections = predictions[arm]
    assert detections and {row["image_id"] for row in detections} <= set(ids)
    (output / f"prediksi_{arm}.json").write_text(json.dumps(detections))
    (output / f"latensi_{arm}.json").write_text(json.dumps(latency[arm]))
    times = [entry["ms"] for entry in latency[arm]]
    assert len(times) == 100 and min(times) > 0
    precision, recall = precision_recall_fixed(gt, detections)
    rows.append({
        "mode": arm, "images": len(ids), "gt_boxes": len(gt.anns),
        **score(gt, detections, ids),
        "precision_conf_0.05_iou_0.5": precision, "recall_conf_0.05_iou_0.5": recall,
        "cpu_median_ms": float(np.median(times)),
        "cpu_p90_ms": float(np.percentile(times, 90)),
        "cpu_p95_ms": float(np.percentile(times, 95)),
        "predictions": len(detections),
    })
table = pd.DataFrame(rows)
table.to_csv(output / "hasil_e4_revisi.csv", index=False)
display(table)
for arm in ARM_NAMES[1:]:
    folder = output / arm
    folder.mkdir()
    details = write_evidence(
        gt, {"full": (predictions["full_native_1280"], []), "sahi": (predictions[arm], [])},
        native / "640/images/val", folder,
    )
    (folder / "contoh.json").write_text(json.dumps(details))
    display(Image(filename=str(folder / "bukti_e4_sahi.png")))
proof = {
    "experiment": "E4 revised",
    "activity": ["Kaggle CPU inference", "COCO validation", "Kaggle CPU latency"],
    "checkpoint_origin": "Historical native E2 from five epochs",
    "checkpoint_sha256": E2_SHA256, "training_performed_on_kaggle": False,

    "native_audit_sha256": sha256(audits[0]), "ground_truth_sha256": sha256(gt_path),
    "source_commit": SOURCE_COMMIT, "source_sha256": code_hashes,
    "images": len(ids), "gt_boxes": len(gt.anns),
    "slice": {
        "tile": TILE, "overlap": OVERLAP, "batch": SLICE_BATCH, "merge_iou": MERGE_IOU,
        "confidence_ap": CONFIDENCE, "max_detections_per_image": 100,
        "full_input_1280": 1280, "standard_inside_sahi_input": 640,
        "tiles_per_image": len(get_slice_bboxes(
            image_height=1280, image_width=1280, slice_height=TILE, slice_width=TILE,
            overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP,
        )),
    },
    "cpu_benchmark": {"logical_cpu_affinity": affinity, "threads": 4, "warmup": 10, "images": 100, "sample_seed": 42},
    "python": sys.version.split()[0],
    "cpu": next(line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines() if line.startswith("model name")),
    "versions": {p: version(p) for p in ("ultralytics", "pycocotools", "sahi", "torch", "torchvision")},
    "outputs_sha256": {str(path.relative_to(output)): sha256(path) for path in sorted(output.rglob("*")) if path.is_file()},
}
(output / "run_e4_revisi.json").write_text(json.dumps(proof, indent=2) + "\n")
print("Bukti E4 tersimpan pada", output)